# A/B Analysis Notebook

This notebook loads A/B assignment and outcomes, computes churn rates by group, runs statistical tests, and produces visualizations. It also includes simple CI-style checks (assertions) that can be run in CI to validate experiment sanity.

In [ ]:
# Imports
import json
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.stats.proportion import proportions_ztest

sns.set(style="whitegrid")

In [ ]:
# Paths (adjust if necessary)
BASE = Path("reports")
ASSIGN = BASE / "ab_assignment.csv"
OUTCOMES = BASE / "ab_outcomes.csv"  # should contain CustomerID,churn (0/1)
EXPOSURES = BASE / "exposures.jsonl"

print("Files:")
print(ASSIGN, ASSIGN.exists())
print(OUTCOMES, OUTCOMES.exists())
print(EXPOSURES, EXPOSURES.exists())

In [ ]:
# Load data (robust to different column names)
assign_df = pd.read_csv(ASSIGN)
out_df = (
    pd.read_csv(OUTCOMES)
    if OUTCOMES.exists()
    else pd.DataFrame(columns=["CustomerID", "churn"])
)

# normalize column names
if "customerID" in assign_df.columns and "CustomerID" not in assign_df.columns:
    assign_df = assign_df.rename(columns={"customerID": "CustomerID"})
if "customerID" in out_df.columns and "CustomerID" not in out_df.columns:
    out_df = out_df.rename(columns={"customerID": "CustomerID"})

df = assign_df.merge(out_df, on="CustomerID", how="left")
df["churn"] = df["churn"].fillna(0).astype(int)
df.head()

In [ ]:
# Summary rates by group
summary = df.groupby("ab_group").agg(n=("CustomerID", "size"), churns=("churn", "sum"))
summary["rate"] = summary["churns"] / summary["n"]
summary.reset_index(inplace=True)
summary

In [ ]:
# Z-test for difference in proportions (A vs B)
if set(["A", "B"]).issubset(
    summary["ab_group"]
    if "ab_group" in summary.columns
    else summary["index"] if "index" in summary.columns else summary.index
):
    # robust selection
    try:
        count = [
            int(summary.loc[summary["ab_group"] == "A", "churns"].values[0]),
            int(summary.loc[summary["ab_group"] == "B", "churns"].values[0]),
        ]
        nobs = [
            int(summary.loc[summary["ab_group"] == "A", "n"].values[0]),
            int(summary.loc[summary["ab_group"] == "B", "n"].values[0]),
        ]
        stat, pval = proportions_ztest(count, nobs)
        print(f"z-stat: {stat:.4f}, p-value: {pval:.4e}")
    except Exception as e:
        print("Could not run z-test:", e)
else:
    print("Both A and B groups not present in summary; cannot perform z-test")

In [ ]:
# Visualization: churn rates by group
plt.figure(figsize=(6, 4))
sns.barplot(x=summary.index, y="rate", data=summary.reset_index())
plt.xticks([0, 1], summary["ab_group"])
plt.ylabel("Churn Rate")
plt.title("Churn Rate by A/B Group")
plt.show()

## CI-style Checks
The next cell runs assertions useful for CI: minimum sample size per group and that data is consistent (no negative churns).
Adjust thresholds as required by your experiment design.

In [ ]:
# CI Checks
MIN_PER_GROUP = 200  # example threshold; override in CI as needed
for _, row in summary.iterrows():
    assert row["n"] >= 1, "Group has zero members"
    assert row["churns"] >= 0, "Negative churns found"

# warn if below recommended threshold
for _, row in summary.iterrows():
    if row["n"] < MIN_PER_GROUP:
        print(
            f"Warning: group {row.get('ab_group', '<unknown>')} has only {row['n']} observations, below recommended {MIN_PER_GROUP}"
        )

# (Optional) check exposures file too
if EXPOSURES.exists():
    with open(EXPOSURES, "r") as f:
        exposures = [json.loads(line) for line in f]
    print(f"Loaded {len(exposures)} exposure records")
    exposures[:5]
else:
    print("No exposures file found")

In [ ]:
# Notebook complete: run all cells to perform A/B analysis
# Additional checks or visualizations can be added below

In [ ]:
# End of notebook
# You can extend analysis, add further statistics, or convert to markdown for reporting.

In [ ]:
# (Optional) additional cells can go here